# Standalone ETL — Transform and Generate Cypher (No Neptune Required)

This notebook demonstrates document-graph's **standalone ETL path** — transforming structured data into typed graph nodes and generating openCypher statements without requiring a Neptune connection or lexical-graph.

Use this when you want to:
- Preview the graph that would be created before writing to any database
- Generate Cypher for execution in a separate pipeline (e.g., batch job, CI/CD)
- Validate transformations locally without cloud dependencies
- Export Cypher statements for use with Neo4j, FalkorDB, or other openCypher databases

**Prerequisites:**
- `pip install graphrag-toolkit-document-graph` (base install, no extras needed)
- No Neptune endpoint required
- No AWS credentials required

## Step 1: Extract — Read Source Data

In [ ]:
import csv

CSV_PATH = 'data/users.csv'

with open(CSV_PATH) as f:
    records = list(csv.DictReader(f))

print(f'Read {len(records)} records from {CSV_PATH}')
print(f'Columns: {list(records[0].keys())}')
print(f'\nSample record:')
records[0]

## Step 2: Transform — Rows to Typed Nodes

Convert each row into a `Node` object with a label, ID, and properties.
No database connection needed — this is pure data transformation.

In [ ]:
from graphrag_toolkit.document_graph import Node, Edge

nodes = []
for record in records:
    node = Node(
        id=record.get('id', record.get('user_id', '')),
        labels=['User'],
        properties={k: v for k, v in record.items() if k != 'id'}
    )
    nodes.append(node)

print(f'Created {len(nodes)} Node objects')
print(f'\nFirst node:')
print(f'  id:         {nodes[0].id}')
print(f'  labels:     {nodes[0].labels}')
print(f'  properties: {nodes[0].properties}')

## Step 3: Generate Cypher — Single Nodes

Generate openCypher MERGE statements for each node. These can be executed against
any openCypher-compatible database (Neptune, Neo4j, FalkorDB, Memgraph).

In [ ]:
from graphrag_toolkit.document_graph.graph_build.cypher_builder import node_to_cypher

# Without tenant (default — labels are backtick-quoted as-is)
print('=== Default tenant (no scoping) ===\n')
for node in nodes[:2]:
    cypher, params = node_to_cypher(node)
    print(f'Cypher: {cypher}')
    print(f'Params: {params}')
    print()

## Step 4: Generate Cypher — With Tenant Scoping

When you specify a `tenant_id`, labels are formatted to match lexical-graph's
multi-tenancy convention: `` `Label{tenant_id}__` ``. This ensures document-graph
nodes coexist cleanly with lexical-graph nodes in the same Neptune cluster.

In [ ]:
TENANT = 'demo'

print(f'=== Tenant "{TENANT}" (scoped labels) ===\n')
for node in nodes[:2]:
    cypher, params = node_to_cypher(node, tenant_id=TENANT)
    print(f'Cypher: {cypher}')
    print(f'Params: {params}')
    print()

## Step 5: Generate Cypher — Batched UNWIND

For bulk writes, `batch_nodes_unwind` generates a single UNWIND query that writes
all nodes in one round-trip. This is significantly faster than individual MERGE
statements for large datasets.

In [ ]:
from graphrag_toolkit.document_graph.graph_build.cypher_builder import batch_nodes_unwind

cypher, params = batch_nodes_unwind(nodes, tenant_id=TENANT)

print(f'Batch query ({len(nodes)} nodes in one statement):\n')
print(f'Cypher: {cypher}')
print(f'\nBatch size: {len(params["batch"])} rows')
print(f'First row:  {params["batch"][0]}')

## Step 6: Generate Cypher — Edges

Create edges between nodes and generate the corresponding Cypher.

In [ ]:
from graphrag_toolkit.document_graph.graph_build.cypher_builder import edge_to_cypher, batch_edges_unwind

# Create some edges
edges = []
if len(nodes) >= 2:
    edges.append(Edge(
        id='e1',
        source_id=nodes[0].id,
        target_id=nodes[1].id,
        label='KNOWS',
        properties={'since': '2024'}
    ))

print('=== Individual edge ===' )
for edge in edges:
    cypher, params = edge_to_cypher(edge)
    print(f'Cypher: {cypher}')
    print(f'Params: {params}')
    print()

if edges:
    print('=== Batched edges ===')
    cypher, params = batch_edges_unwind(edges)
    print(f'Cypher: {cypher}')
    print(f'Batch:  {params["batch"]}')

## Step 7: Export Cypher to File

Write all generated Cypher statements to a file for execution elsewhere —
useful for batch jobs, code review, or loading into a different graph database.

In [ ]:
output_path = 'data/generated_cypher.txt'

with open(output_path, 'w') as f:
    f.write(f'// Generated by document-graph standalone ETL\n')
    f.write(f'// Tenant: {TENANT}\n')
    f.write(f'// Nodes: {len(nodes)}, Edges: {len(edges)}\n\n')

    for node in nodes:
        cypher, params = node_to_cypher(node, tenant_id=TENANT)
        # Inline params for standalone execution
        stmt = cypher.replace('$id_val', repr(params['id_val']))
        f.write(f'{stmt};\n')

    for edge in edges:
        cypher, params = edge_to_cypher(edge)
        stmt = cypher.replace('$src_id', repr(params['src_id'])).replace('$tgt_id', repr(params['tgt_id']))
        f.write(f'{stmt};\n')

print(f'✅ Exported {len(nodes)} node + {len(edges)} edge statements to {output_path}')
print(f'\nPreview:')
with open(output_path) as f:
    for line in f.readlines()[:8]:
        print(f'  {line.rstrip()}')

## Summary

This notebook used **only** `graphrag-toolkit-document-graph` (base install) — no Neptune, no lexical-graph, no AWS credentials.

| What we did | Dependency required |
|-------------|--------------------|
| Read CSV | None (stdlib) |
| Create Node/Edge objects | `graphrag-toolkit-document-graph` |
| Generate Cypher (single + batch) | `graphrag-toolkit-document-graph` |
| Tenant-scoped labels | `graphrag-toolkit-document-graph` (falls back to standalone format) |
| Export to file | None (stdlib) |

To actually **execute** these statements against Neptune, see `01-Combined-Extract-Load-Build.ipynb`
(requires `[graphrag]` extra for Neptune connectivity).